# 🐶 DogDex — Dog Breed Classifier Training

This notebook trains an **EfficientNetV2-S** image classifier on **120 dog breeds** using the [Stanford Dogs Dataset](http://vision.stanford.edu/aditya86/ImageNetDogs/), then exports a `.tflite` model file ready for on-device inference in the DogDex mobile app.

---

## What this notebook does

1. **Downloads** the Stanford Dogs Dataset automatically (no manual download needed)
2. **Trains** EfficientNetV2-S — a state-of-the-art image classifier — in two phases:
   - Phase 1 (15 epochs, ~20 min): Trains only the new classification head while the base model is frozen
   - Phase 2 (up to 20 epochs, ~60–80 min): Fine-tunes the top layers of the base model at a lower learning rate
3. **Evaluates** the model on a held-out test set and shows sample predictions
4. **Exports** two files:
   - `dog_breed_classifier.tflite` (~85 MB) — the model file for the app
   - `labels.json` — maps model output index → breed name string
5. **Saves** both files to your Google Drive so you don't lose them when the Colab session ends

## How long does it take?

On a **free Colab T4 GPU**: approximately **1.5–2 hours** total.

> ⚠️ **Important:** Free Colab sessions disconnect after ~90 minutes of idle time. The notebook saves checkpoints automatically, but try to keep the tab active. If it disconnects, re-run from the checkpoint cell.

## What you need before starting

- A Google account (for Google Drive saving)
- A free Google Colab account at [colab.research.google.com](https://colab.research.google.com)
- **GPU runtime enabled:** Runtime → Change runtime type → T4 GPU

## Using the output files

Once training is complete:
1. Download `dog_breed_classifier.tflite` and `labels.json` from your Google Drive (or from the Colab file browser on the left)
2. Place them in `artifacts/mobile/assets/ml/` in the DogDex project
3. The integration task ("Replace GPT calls with on-device TFLite breed detection") will wire them into the app

## Expected accuracy

EfficientNetV2-S fine-tuned on Stanford Dogs typically achieves **88–92% top-1 accuracy** on the validation split. The target for this notebook is ≥85%.

---

*Run all cells top-to-bottom: Runtime → Run all*

---
## Section 1 — Setup

Install required packages and verify the GPU is available.

In [ ]:
# Install / upgrade packages
# tensorflow-datasets provides the Stanford Dogs dataset with automatic download
!pip install -q --upgrade tensorflow tensorflow-datasets numpy matplotlib

In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds
import numpy as np
import matplotlib.pyplot as plt
import json
import os
import shutil
from datetime import datetime

print(f"TensorFlow version: {tf.__version__}")
print(f"TF-Datasets version: {tfds.__version__}")

# Verify GPU
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"\n✅ GPU available: {gpus[0].name}")
    # Allow GPU memory to grow incrementally instead of grabbing it all at once
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print("   Memory growth enabled")
else:
    print("\n⚠️  No GPU detected!")
    print("   Go to: Runtime → Change runtime type → Hardware accelerator → T4 GPU")
    print("   Then re-run all cells.")

# Output directory for model checkpoints and exports
OUTPUT_DIR = "/content/dogdex_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"\nOutput directory: {OUTPUT_DIR}")

---
## Section 2 — Dataset

Load the Stanford Dogs dataset (120 breeds, ~20,000 images). `tensorflow_datasets` downloads and caches it automatically — no manual setup needed.

We apply data augmentation to the training split to help the model generalise better and reduce overfitting.

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
IMG_SIZE      = 384          # EfficientNetV2-S recommended input size
BATCH_SIZE    = 32           # Fits comfortably on a T4 GPU
NUM_CLASSES   = 120          # Stanford Dogs has exactly 120 breeds
AUTOTUNE      = tf.data.AUTOTUNE

# Train/val/test proportions (Stanford Dogs only ships train+test splits,
# so we carve our own val set out of the official training split)
# We use: 80% train | 10% val | 10% test (from the full dataset)
TRAIN_SPLIT = "train[:80%]"
VAL_SPLIT   = "train[80%:90%]"
TEST_SPLIT  = "train[90%:]"

print(f"Image size : {IMG_SIZE}×{IMG_SIZE}")
print(f"Batch size : {BATCH_SIZE}")
print(f"Num classes: {NUM_CLASSES}")

In [ ]:
# Load Stanford Dogs dataset
# First call downloads ~800 MB — this may take 5–10 minutes
print("Downloading Stanford Dogs dataset (this may take a few minutes)...")

(ds_train_raw, ds_val_raw, ds_test_raw), ds_info = tfds.load(
    "stanford_dogs",
    split=[TRAIN_SPLIT, VAL_SPLIT, TEST_SPLIT],
    as_supervised=False,   # Returns dict with 'image' and 'label'
    with_info=True,
    shuffle_files=True,
)

print(f"\n✅ Dataset loaded")
print(f"   Train batches : {len(ds_train_raw):,} examples")
print(f"   Val batches   : {len(ds_val_raw):,} examples")
print(f"   Test batches  : {len(ds_test_raw):,} examples")

# Build a sorted list of breed names (index 0..119 → breed name)
# tfds encodes labels as integers; get_label_names() returns them in sorted order
label_names = ds_info.features['label'].names
print(f"\nSample breed labels:")
for i in [0, 10, 50, 100, 119]:
    print(f"  [{i:3d}] {label_names[i]}")

In [ ]:
# ── Preprocessing & augmentation helpers ────────────────────────────────────

def preprocess(example, augment=False):
    """Resize, normalise, and optionally augment a single example."""
    image = tf.cast(example['image'], tf.float32)
    label = example['label']

    # Resize to the model's expected input resolution
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])

    if augment:
        # Random horizontal flip (dogs look like dogs both ways)
        image = tf.image.random_flip_left_right(image)
        # Random brightness ±20% — simulates different lighting conditions
        image = tf.image.random_brightness(image, max_delta=0.2)
        # Random saturation — helps with varied photo quality
        image = tf.image.random_saturation(image, lower=0.8, upper=1.2)
        # Random rotation ±15° using affine transform
        image = tf.keras.layers.RandomRotation(0.042)(tf.expand_dims(image, 0))[0]
        # Clip to valid pixel range after augmentation
        image = tf.clip_by_value(image, 0.0, 255.0)

    # EfficientNetV2 expects pixel values in [0, 255] — it handles normalisation internally
    label = tf.one_hot(label, NUM_CLASSES)
    return image, label


def build_dataset(ds_raw, augment=False, shuffle=False, cache=True):
    """Apply preprocessing and build an optimised tf.data pipeline."""
    ds = ds_raw.map(lambda ex: preprocess(ex, augment=augment), num_parallel_calls=AUTOTUNE)
    if cache:
        ds = ds.cache()   # Cache in memory after first epoch — speeds up training significantly
    if shuffle:
        ds = ds.shuffle(buffer_size=1000, seed=42)
    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(AUTOTUNE)  # Prefetch next batch while GPU trains on current batch
    return ds


ds_train = build_dataset(ds_train_raw, augment=True,  shuffle=True,  cache=True)
ds_val   = build_dataset(ds_val_raw,   augment=False, shuffle=False, cache=True)
ds_test  = build_dataset(ds_test_raw,  augment=False, shuffle=False, cache=True)

print("✅ Data pipelines built")
print(f"   Each batch shape: {next(iter(ds_train))[0].shape}  (batch × H × W × C)")

In [ ]:
# Quick sanity check — show a grid of training samples
images, labels = next(iter(ds_train))

fig, axes = plt.subplots(3, 6, figsize=(18, 9))
fig.suptitle("Sample augmented training images", fontsize=14)
for i, ax in enumerate(axes.flat):
    img = images[i].numpy().astype('uint8')
    breed_idx = tf.argmax(labels[i]).numpy()
    breed_name = label_names[breed_idx].replace('-', ' ').title()
    ax.imshow(img)
    ax.set_title(breed_name, fontsize=7)
    ax.axis('off')
plt.tight_layout()
plt.show()

---
## Section 3 — Model

We use **EfficientNetV2-S** pretrained on ImageNet as our base, then add a lightweight classification head for 120 dog breeds.

**Why EfficientNetV2-S?**
- State-of-the-art accuracy vs. speed tradeoff
- Compact enough (~85 MB as TFLite) for mobile deployment
- ImageNet pretraining means it already "knows" what dogs look like — we just need to specialise it

In [ ]:
def build_model(num_classes=NUM_CLASSES, dropout_rate=0.3):
    """
    Build EfficientNetV2-S with a custom classification head.

    Architecture:
      Input (384×384×3)
      → EfficientNetV2-S base (pretrained on ImageNet, frozen initially)
      → GlobalAveragePooling2D  (collapses spatial dims into a feature vector)
      → Dropout(0.3)            (regularisation to prevent overfitting)
      → Dense(120, softmax)     (one output per breed)
    """
    # Load EfficientNetV2-S without the top classification layer
    # include_top=False lets us add our own head for 120 classes
    base_model = tf.keras.applications.EfficientNetV2S(
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        include_top=False,
        weights='imagenet',   # Start with ImageNet weights
    )

    # Freeze all base layers — only the head will train in Phase 1
    base_model.trainable = False

    # Build the full model
    inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name='input_image')

    # EfficientNetV2 includes its own preprocessing — pass images in [0, 255]
    x = base_model(inputs, training=False)

    # Global average pooling collapses (H × W × C) → (C,)
    x = tf.keras.layers.GlobalAveragePooling2D(name='avg_pool')(x)

    # Dropout prevents the head from memorising training examples
    x = tf.keras.layers.Dropout(dropout_rate, name='head_dropout')(x)

    # Final classification layer — one logit per breed, softmax for probabilities
    outputs = tf.keras.layers.Dense(
        num_classes, activation='softmax', name='predictions'
    )(x)

    model = tf.keras.Model(inputs, outputs, name='dogdex_efficientnetv2s')
    return model, base_model


model, base_model = build_model()

# Compile for Phase 1 — higher learning rate while base is frozen
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy', tf.keras.metrics.TopKCategoricalAccuracy(k=5, name='top5_accuracy')],
)

total_params     = model.count_params()
trainable_params = sum([tf.size(w).numpy() for w in model.trainable_weights])

print(f"✅ Model built")
print(f"   Total parameters     : {total_params:,}")
print(f"   Trainable parameters : {trainable_params:,}  (head only — base is frozen)")
model.summary(line_length=90, expand_nested=False)

---
## Section 4 — Training

Training happens in **two phases**:

| Phase | Epochs | What trains | Learning rate | Purpose |
|-------|--------|-------------|--------------|----------|
| 1 | 15 | Head only | 1e-3 | Fast warmup — learn to classify without disturbing base weights |
| 2 | up to 20 | Top 30 base layers + head | 1e-4 | Fine-tune — adapt base features to dog breeds |

**EarlyStopping** monitors validation accuracy and halts automatically if training plateaus (patience = 5 epochs), saving time.

In [ ]:
# ── Phase 1: Head warmup (base frozen) ───────────────────────────────────────
PHASE1_EPOCHS = 15

checkpoint_path = os.path.join(OUTPUT_DIR, "best_model.weights.h5")

callbacks_phase1 = [
    # Save the best weights whenever validation accuracy improves
    tf.keras.callbacks.ModelCheckpoint(
        filepath=checkpoint_path,
        monitor='val_accuracy',
        save_best_only=True,
        save_weights_only=True,
        verbose=1,
    ),
    # Stop early if val_accuracy doesn't improve for 5 consecutive epochs
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=5,
        restore_best_weights=True,
        verbose=1,
    ),
    # Reduce learning rate when training stalls — helps squeeze out more accuracy
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1,
    ),
]

print("Starting Phase 1: Head warmup (base frozen)")
print(f"Running for up to {PHASE1_EPOCHS} epochs...\n")

history_phase1 = model.fit(
    ds_train,
    epochs=PHASE1_EPOCHS,
    validation_data=ds_val,
    callbacks=callbacks_phase1,
    verbose=1,
)

val_acc = max(history_phase1.history['val_accuracy'])
print(f"\n✅ Phase 1 complete — best val accuracy: {val_acc:.1%}")

In [ ]:
# ── Phase 2: Fine-tune top layers of the base model ──────────────────────────
PHASE2_EPOCHS     = 20
UNFREEZE_LAYERS   = 30   # Unfreeze the last 30 layers of EfficientNetV2-S
FINETUNE_LR       = 1e-4 # Much lower LR to avoid destroying pretrained weights

# Unfreeze the entire base first, then re-freeze everything except the top N layers
base_model.trainable = True
total_layers = len(base_model.layers)
freeze_until = total_layers - UNFREEZE_LAYERS

for layer in base_model.layers[:freeze_until]:
    layer.trainable = False

trainable_now = sum(1 for l in base_model.layers if l.trainable)
print(f"Base model layers    : {total_layers}")
print(f"Layers unfrozen      : {trainable_now} (top {UNFREEZE_LAYERS})")
print(f"Fine-tune LR         : {FINETUNE_LR}")

# Recompile with lower learning rate for fine-tuning
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=FINETUNE_LR),
    loss='categorical_crossentropy',
    metrics=['accuracy', tf.keras.metrics.TopKCategoricalAccuracy(k=5, name='top5_accuracy')],
)

# Fine-tune checkpoint saves over Phase 1 if we get better results
callbacks_phase2 = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=checkpoint_path,
        monitor='val_accuracy',
        save_best_only=True,
        save_weights_only=True,
        verbose=1,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=5,
        restore_best_weights=True,
        verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1,
    ),
]

# Offset the epoch counter so plots show a continuous timeline
initial_epoch = len(history_phase1.history['accuracy'])

print(f"\nStarting Phase 2: Fine-tuning top {UNFREEZE_LAYERS} layers")
print(f"Running for up to {PHASE2_EPOCHS} epochs (starting from epoch {initial_epoch})...\n")

history_phase2 = model.fit(
    ds_train,
    epochs=initial_epoch + PHASE2_EPOCHS,
    initial_epoch=initial_epoch,
    validation_data=ds_val,
    callbacks=callbacks_phase2,
    verbose=1,
)

val_acc_phase2 = max(history_phase2.history['val_accuracy'])
print(f"\n✅ Phase 2 complete — best val accuracy: {val_acc_phase2:.1%}")

# Load the best weights saved by ModelCheckpoint
model.load_weights(checkpoint_path)
print(f"Best weights restored from: {checkpoint_path}")

---
## Section 5 — Evaluation

Visualise training progress and evaluate the final model on the held-out test set.

In [ ]:
# ── Training curves ──────────────────────────────────────────────────────────

# Merge Phase 1 and Phase 2 histories
def merge_history(h1, h2):
    merged = {}
    for key in h1.history:
        merged[key] = h1.history[key] + h2.history.get(key, [])
    return merged

history = merge_history(history_phase1, history_phase2)
epochs  = range(1, len(history['accuracy']) + 1)
p1_end  = len(history_phase1.history['accuracy'])  # Where Phase 1 ends

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("DogDex — EfficientNetV2-S Training History", fontsize=14, fontweight='bold')

# Accuracy plot
ax1.plot(epochs, history['accuracy'],     label='Train accuracy',      color='steelblue')
ax1.plot(epochs, history['val_accuracy'], label='Validation accuracy', color='coral',    linestyle='--')
ax1.axvline(x=p1_end, color='gray', linestyle=':', linewidth=1.5, label='Phase 1 → 2')
ax1.set_title('Accuracy')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))

# Loss plot
ax2.plot(epochs, history['loss'],     label='Train loss',      color='steelblue')
ax2.plot(epochs, history['val_loss'], label='Validation loss', color='coral', linestyle='--')
ax2.axvline(x=p1_end, color='gray', linestyle=':', linewidth=1.5, label='Phase 1 → 2')
ax2.set_title('Loss')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Training curves saved to output directory")

In [ ]:
# ── Test set evaluation ──────────────────────────────────────────────────────

print("Evaluating on held-out test split...")
test_loss, test_acc, test_top5 = model.evaluate(ds_test, verbose=1)

print(f"\n{'='*45}")
print(f"  Test Loss     : {test_loss:.4f}")
print(f"  Top-1 Accuracy: {test_acc:.1%}")
print(f"  Top-5 Accuracy: {test_top5:.1%}")
print(f"{'='*45}")

TARGET_ACCURACY = 0.85
if test_acc >= TARGET_ACCURACY:
    print(f"\n✅ Target accuracy met ({TARGET_ACCURACY:.0%} threshold)")
else:
    print(f"\n⚠️  Below target ({TARGET_ACCURACY:.0%}). Consider:")
    print("    - More fine-tune epochs (increase PHASE2_EPOCHS)")
    print("    - Unfreeze more base layers (increase UNFREEZE_LAYERS to 50)")
    print("    - Use a larger model (EfficientNetV2-M for ~+2% accuracy)")

In [ ]:
# ── Sample predictions grid ───────────────────────────────────────────────────
# Visual sanity check — shows what the model predicts on real test images

images_batch, labels_batch = next(iter(ds_test))
predictions = model.predict(images_batch, verbose=0)

n_show = 18
fig, axes = plt.subplots(3, 6, figsize=(20, 10))
fig.suptitle("Sample Predictions (green = correct, red = wrong)", fontsize=13, fontweight='bold')

for i, ax in enumerate(axes.flat):
    if i >= n_show:
        ax.axis('off')
        continue

    img           = images_batch[i].numpy().astype('uint8')
    true_idx      = tf.argmax(labels_batch[i]).numpy()
    pred_idx      = tf.argmax(predictions[i]).numpy()
    confidence    = predictions[i][pred_idx]

    true_name = label_names[true_idx].split('-', 1)[-1].replace('_', ' ').title()
    pred_name = label_names[pred_idx].split('-', 1)[-1].replace('_', ' ').title()
    is_correct = true_idx == pred_idx

    ax.imshow(img)
    color = 'green' if is_correct else 'red'
    title = f"✓ {pred_name}" if is_correct else f"✗ {pred_name}\n(true: {true_name})"
    ax.set_title(f"{title}\n{confidence:.0%}", fontsize=7, color=color)
    ax.axis('off')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'sample_predictions.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## Section 6 — Export

Convert the trained model to **TFLite format** for on-device inference, and export the breed label mapping as `labels.json`.

Two variants are exported:
- **Float32** (default, ~85 MB) — Full precision, maximum accuracy
- **INT8 quantized** (optional, ~22 MB) — 4× smaller, minimal accuracy loss (<1%)

In [ ]:
# ── Export labels.json ───────────────────────────────────────────────────────
# Maps integer index → breed name string
# The mobile app uses this to turn model output [0.01, 0.89, ...] into "Golden Retriever"

# Clean up the raw TFDS label names (e.g. 'n02085620-Chihuahua' → 'Chihuahua')
def clean_breed_name(raw_name):
    """Strip the WordNet synset prefix (e.g. 'n02085620-') and format nicely."""
    name = raw_name.split('-', 1)[-1]   # Remove 'nXXXXXXXX-' prefix if present
    name = name.replace('_', ' ')       # Replace underscores with spaces
    # Title-case while preserving short words
    return ' '.join(word.capitalize() for word in name.split())

labels_dict = {
    str(i): clean_breed_name(label_names[i])
    for i in range(len(label_names))
}

labels_path = os.path.join(OUTPUT_DIR, "labels.json")
with open(labels_path, 'w') as f:
    json.dump(labels_dict, f, indent=2, sort_keys=False)

print(f"✅ labels.json saved → {labels_path}")
print(f"   {len(labels_dict)} breed labels")
print("\nSample entries:")
for k in ['0', '10', '50', '100', '119']:
    print(f"  {k:>3}: {labels_dict[k]}")

In [ ]:
# ── Export Float32 TFLite model ───────────────────────────────────────────────
# This is the primary export — full precision, best accuracy

print("Converting to TFLite (float32)...")
print("This may take 3–5 minutes...\n")

converter = tf.lite.TFLiteConverter.from_keras_model(model)
# Default settings = float32, no quantisation — maximum accuracy
tflite_model = converter.convert()

tflite_path = os.path.join(OUTPUT_DIR, "dog_breed_classifier.tflite")
with open(tflite_path, 'wb') as f:
    f.write(tflite_model)

size_mb = os.path.getsize(tflite_path) / 1e6
print(f"✅ Float32 TFLite model saved")
print(f"   Path : {tflite_path}")
print(f"   Size : {size_mb:.1f} MB")

In [ ]:
# ── Verify the TFLite model runs correctly ───────────────────────────────────
# Run a quick inference test to make sure the export is valid

print("Verifying TFLite model with a test inference...")

# Load the interpreter
interpreter = tf.lite.Interpreter(model_path=tflite_path)
interpreter.allocate_tensors()

input_details  = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print(f"Input  shape : {input_details[0]['shape']}  dtype: {input_details[0]['dtype']}")
print(f"Output shape : {output_details[0]['shape']} dtype: {output_details[0]['dtype']}")

# Run one inference
sample_image, sample_label = next(iter(ds_test.unbatch().batch(1)))
interpreter.set_tensor(input_details[0]['index'], sample_image.numpy())
interpreter.invoke()
output = interpreter.get_tensor(output_details[0]['index'])

pred_idx   = np.argmax(output[0])
pred_conf  = output[0][pred_idx]
true_idx   = tf.argmax(sample_label[0]).numpy()
pred_breed = labels_dict[str(pred_idx)]
true_breed = labels_dict[str(true_idx)]

match = "✅" if pred_idx == true_idx else "⚠️"
print(f"\n{match} Inference test")
print(f"   Predicted : {pred_breed} ({pred_conf:.1%} confidence)")
print(f"   True label: {true_breed}")
print("\n✅ TFLite model verified and ready for deployment!")

In [ ]:
# ── (Optional) Export INT8 quantised TFLite model ────────────────────────────
#
# INT8 quantisation reduces the model from ~85 MB to ~22 MB at a small
# accuracy cost (typically < 1% top-1 accuracy drop).
#
# Use this if:
#   - You want to reduce app download size
#   - You're targeting lower-end Android devices
#
# The float32 model is preferred for maximum accuracy.
#
# To skip INT8 export, you can skip running this cell.

print("Converting to INT8 quantised TFLite model...")
print("This requires a representative dataset sample for calibration (~5 min)...\n")

def representative_dataset_gen():
    """Yields batches of images for quantisation calibration (100 batches)."""
    for i, (images, _) in enumerate(ds_val.unbatch().batch(1)):
        yield [images.numpy()]
        if i >= 100:
            break

converter_int8 = tf.lite.TFLiteConverter.from_keras_model(model)
converter_int8.optimizations = [tf.lite.Optimize.DEFAULT]
converter_int8.representative_dataset = representative_dataset_gen
converter_int8.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter_int8.inference_input_type  = tf.float32  # Keep float input for ease of use
converter_int8.inference_output_type = tf.float32  # Keep float output

tflite_int8 = converter_int8.convert()

int8_path = os.path.join(OUTPUT_DIR, "dog_breed_classifier_int8.tflite")
with open(int8_path, 'wb') as f:
    f.write(tflite_int8)

size_float32 = os.path.getsize(tflite_path) / 1e6
size_int8    = os.path.getsize(int8_path) / 1e6

print(f"✅ INT8 quantised model saved")
print(f"   Path         : {int8_path}")
print(f"   Float32 size : {size_float32:.1f} MB")
print(f"   INT8 size    : {size_int8:.1f} MB  ({size_int8/size_float32:.0%} of original)")
print(f"\nNote: Use 'dog_breed_classifier.tflite' (float32) in the app unless")
print(f"      size is a critical concern. INT8 saves ~{size_float32-size_int8:.0f} MB.")

---
## Section 7 — Save to Google Drive

Colab sessions reset when they end — **all files are deleted**. This cell copies the model files to your Google Drive so they're safe.

You'll be prompted to authorise access to your Drive the first time.

In [ ]:
# ── Mount Google Drive and save output files ──────────────────────────────────

from google.colab import drive

print("Mounting Google Drive...")
print("(A permission prompt will appear — click 'Connect to Google Drive')\n")

drive.mount('/content/drive')

# Create a dedicated folder in Drive
drive_folder = "/content/drive/MyDrive/DogDex_ML_Model"
os.makedirs(drive_folder, exist_ok=True)

# Files to copy
files_to_save = {
    tflite_path  : os.path.join(drive_folder, "dog_breed_classifier.tflite"),
    labels_path  : os.path.join(drive_folder, "labels.json"),
    os.path.join(OUTPUT_DIR, 'training_curves.png')    : os.path.join(drive_folder, 'training_curves.png'),
    os.path.join(OUTPUT_DIR, 'sample_predictions.png') : os.path.join(drive_folder, 'sample_predictions.png'),
}

# Optionally include the INT8 model if it was exported
int8_path_check = os.path.join(OUTPUT_DIR, "dog_breed_classifier_int8.tflite")
if os.path.exists(int8_path_check):
    files_to_save[int8_path_check] = os.path.join(drive_folder, "dog_breed_classifier_int8.tflite")

print(f"Saving files to Google Drive: {drive_folder}")
print()

for src, dst in files_to_save.items():
    if os.path.exists(src):
        shutil.copy2(src, dst)
        size = os.path.getsize(dst)
        print(f"  ✅ {os.path.basename(dst)}  ({size/1e6:.1f} MB)")
    else:
        print(f"  ⚠️  Skipped {os.path.basename(src)} (not found)")

print(f"\n🎉 All files saved to Google Drive!")
print(f"   Location: MyDrive/DogDex_ML_Model/")
print(f"\nNext steps:")
print(f"  1. Download 'dog_breed_classifier.tflite' and 'labels.json' from Drive")
print(f"  2. Place them in: artifacts/mobile/assets/ml/")
print(f"  3. The integration task will wire them into the DogDex app")

---
## 🎉 Done!

### What was produced

| File | Description |
|------|-------------|
| `dog_breed_classifier.tflite` | Float32 TFLite model (~85 MB), 120 breed classes |
| `labels.json` | Index → breed name mapping (120 entries) |
| `dog_breed_classifier_int8.tflite` | INT8 quantised variant (~22 MB), optional |
| `training_curves.png` | Accuracy and loss plots across both training phases |
| `sample_predictions.png` | Grid of test predictions with confidence scores |

### Troubleshooting

**Session disconnected mid-training?**  
- Re-run the Setup and Dataset sections first  
- Skip to Phase 2 and load weights from the checkpoint: `model.load_weights(checkpoint_path)`  
- Continue from where you left off

**Accuracy below 85%?**  
- Increase `PHASE2_EPOCHS` to 30 and rerun Phase 2  
- Increase `UNFREEZE_LAYERS` to 50 for deeper fine-tuning  
- Ensure you're using a T4 GPU (Runtime → Change runtime type)

**Out of GPU memory?**  
- Reduce `BATCH_SIZE` from 32 to 16  
- Runtime → Disconnect and delete runtime, then reconnect

**Want higher accuracy?**  
- Switch to `EfficientNetV2-M` (replace `EfficientNetV2S` with `EfficientNetV2M`) — adds ~+1–2% accuracy at the cost of a larger model file